# Editing images - Fast

A typical usage of "kernels" is to edit images. In this example, we will see how to use the `KernelAbstractions` library to edit images in a fast and efficient way.


#### Content
- [Introduction](#introduction)
- [Simple image kernel usage](#simple-image-kernels)
- [Large scale editing - comparison with CPU](#large-scale-editing---comparison-with-cpu)


### Introduction
For one that is not familiar with the usage of kernels for image editing, the idea is to apply a function (the kernel) to each pixel of the image, taking into account its neighbors. This is often used for tasks such as blurring, sharpening, edge detection, etc.

For example, if we want to apply a simple blur to an image, we can use a kernel that takes the average of the pixel and its neighbors. This can be done using a $3\times3$ kernel like this:

$$
\frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1 \\
\end{bmatrix}
$$

Similarly, for edge detection, we can use a kernel like this:
$$
\begin{bmatrix}
0 & -1 & 0 \\
-1 & 4 & -1 \\
0 & -1 & 0 \\
\end{bmatrix}
$$
The idea with the last one is that it will highlight the edges of the image by taking the difference between the pixel and its neighbors. If the pixel is similar to its neighbors, the result will be close to zero, but if it is different, the result will be larger.

For more examples of kernels for image editing, one can check out the [Wikipedia page](https://en.wikipedia.org/wiki/Kernel_(image_processing)) or the [Victor Powell's interactive page](https://setosa.io/ev/image-kernels/).

### Simple image kernels 

We'll first start with a simple example of applying a blur kernel, and an edge detection kernel to an image. We will use the `KernelAbstractions` library to do this in a fast and efficient way.

In [ ]:
using KernelAbstractions, CUDA
using Images, FileIO

backend = CUDABackend() # Specify backend

In [ ]:
# Define helper functions to help translate images to tensors, and back.
function img_to_tensor(img)
    img_rgb = RGB.(img)
    tensor = channelview(img_rgb)
    return Float32.(tensor)
end

function tensor_to_img(tensor)
    @assert size(tensor, 1) == 3 "Tensor must have 3 channels"
    img = colorview(RGB, tensor)
    return img
end

Time to define some kernels!

In [ ]:
@kernel function blur_kernel!(output::AbstractArray, input::AbstractArray)
    k, i, j = @index(Global, NTuple)
    C, H, W = size(input)
    if (1 <=k <=C) && (2 <= i <= H-1) && (2 <= j <= W-1)
        output[k, i, j] = 0.2 * sum(input[k,i,], [input[k,i-1,j], input[k,i,j-1], input[k,i+1,j], input[k,i,j+1]])
    elseif (1 <=k <=C) && (1 <= i <= H) && (1 <= j <= W) #Edges
        output[k,i,j] = input[k,i,j]
    end
end

@kernel function edge_detection_kernel!(output::AbstractArray, input::AbstractArray)
    k, i, j = @index(Global, NTuple)
    C, H, W = size(input)
    if (1 <=k <=C) && (2 <= i <= H-1) && (2 <= j <= W-1)
        output[k, i, j] = 4*input[k,i,j] - sum([input[k,i-1,j], input[k,i,j-1], input[k,i+1,j], input[k,i,j+1]])
    elseif (1 <=k <=C) && (1 <= i <= H) && (1 <= j <= W) #Edges
        output[k,i,j] = input[k,i,j]
    end
end



function sharpenImage!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = sharpening_kernel!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

As has to be done with kernels, we have to define a "wrapper" function that will take care of the boundaries of the image, and specify the backend. 

In [ ]:
function blurImage!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = blur_kernel!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

function edgeDetectImage!(output, img) 
    backend = KernelAbstractions.get_backend(img)
    kernel! = edge_detection_kernel!(backend)
    kernel!(output, img, ndrange=size(img))
    KernelAbstractions.synchronize(backend)
    return output
end

Moreover, let's also define some versions of these functions that will copy the image (Note the lack of exclamation mark). While we are at it we can also define functions that operate directly on the `Images` file, and not the array (#heart multiple dispatch!).

In [ ]:
function blurImage(img::Array{Float32, 3})
    output = similar(img)
    imgC = deepcopy(img)
    blurImage!(output,imgC)
    return output
end

function edgeDetectImage(img::Array{Float32, 3})
    output = similar(img)
    imgC = deepcopy(img)
    edgeDetectImage!(output,imgC)
    return output
end

function blurImage(img::Matrix{RGB{N0f8}})
    img_tensor = img_to_tensor(img)
    output = similar(img_tensor)
    blurImage!(output,img_tensor)
    return tensor_to_img(output)
end

function edgeDetectImage(img::Matrix{RGB{N0f8}})
    img_tensor = img_to_tensor(img)
    output = similar(img_tensor)
    edgeDetectImage!(output,img_tensor)
    return tensor_to_img(output)
end


Now that we have all this boilerplate out of the way, we can apply our kernels to an image and see the results!

Check out this nice image of Phillip (credit to C. Wittens)

In [ ]:
# Load Image
original_phil = load("PhilC.jpg")

Now look at the blurred version

In [ ]:
blurred_phil = blurImage(original_phil)

Cool! 

What about the edge detection?

In [ ]:
phils_edges = edgeDetectImage(original_phil)

### Large scale editing - comparison with CPU
Of course, the above example is not very impressive, since the image is small and the kernels are simple. However, the real power of using kernels for image editing comes when we have many - or large - images.

In [ ]:
using BenchmarkTools